## 10. Outfit Suitability Evaluation

This section evaluates whether the detected outfit is suitable for the selected occasion, time of day, weather conditions, and temperature.

The evaluation starts with a score of 10 and applies penalties when clothing items conflict with the given context.

The system checks:

- **Occasion:** Work, University, Sport, or Party.
- **Weather:** Cold, Hot, Rainy, or Sunny conditions.
- **Temperature:** Determines whether the clothing is appropriate for the current temperature.
- **Time of Day:** Identifies whether items such as sunglasses are appropriate.
- **Detected Clothing:** Uses the clothing items detected by the YOLO model.

The system then generates:
- A **Suitability Score out of 10**.
- Personalized **Feedback and Recommendations** explaining any detected issues.

In [2]:
# ==========================================
# Outfit Suitability Evaluation
# ==========================================

import numpy as np

In [3]:
# ==========================================
# 1. Input Normalization
# ==========================================

def normalize_items(items):

    return [
        str(item).strip().lower()
        for item in items
    ]


def normalize_text(value):

    return str(value).strip().lower()

In [4]:
# ==========================================
# 2. Outfit Suitability Evaluation
# ==========================================

def evaluate_outfit_suitability(
    detected_items,
    occasion,
    time_of_day,
    weather,
    temperature
):
    """
    Evaluate outfit suitability based on:

    - Occasion
    - Time of day
    - Weather
    - Temperature
    - Detected clothing items

    Returns:
        final_score
        feedback
    """

    # Normalize inputs
    items = normalize_items(
        detected_items
    )

    occasion = normalize_text(
        occasion
    )

    time_of_day = normalize_text(
        time_of_day
    )

    weather = normalize_text(
        weather
    )

    temperature = float(
        temperature
    )

    # Start with maximum score
    score = 10.0

    feedback = []

    # ==========================================
    # 1. Occasion Rules
    # ==========================================

    if occasion in {
        "work",
        "university"
    }:

        if any(
            item in items
            for item in [
                "shorts",
                "short_pants",
                "sport_shorts"
            ]
        ):

            score -= 3.5

            feedback.append(
                "Shorts may be too casual for "
                "work or university."
            )

    elif occasion == "sport":

        # Avoid over-penalizing jackets.
        # Some jackets are suitable for sports.

        if any(
            item in items
            for item in [
                "blazer",
                "dress",
                "heels"
            ]
        ):

            score -= 3.0

            feedback.append(
                "Some formal items may restrict "
                "movement during sports activities."
            )

        if "jacket" in items:

            feedback.append(
                "A lightweight sports jacket can "
                "be suitable depending on the activity."
            )

    elif occasion == "party":

        if any(
            item in items
            for item in [
                "tracksuit",
                "sweatpants",
                "sportswear"
            ]
        ):

            score -= 2.0

            feedback.append(
                "Sportswear may look too casual "
                "for some party settings."
            )

    # ==========================================
    # 2. Cold Weather
    # ==========================================

    cold_condition = (
        temperature < 18
        or weather == "cold"
    )

    if cold_condition:

        light_items = [
            "shorts",
            "t-shirt",
            "tank top",
            "skirt"
        ]

        if any(
            item in items
            for item in light_items
        ):

            score -= 2.0

            feedback.append(
                "The outfit may not provide enough "
                "warmth for the current conditions."
            )

        outerwear = [
            "jacket",
            "coat",
            "sweater",
            "hoodie"
        ]

        if not any(
            item in items
            for item in outerwear
        ):

            score -= 1.5

            feedback.append(
                "Consider adding a jacket, coat, "
                "sweater, or hoodie."
            )

    # ==========================================
    # 3. Hot Weather
    # ==========================================

    hot_condition = (
        temperature > 28
        or weather == "hot"
    )

    if hot_condition:

        heavy_items = [
            "coat",
            "heavy_jacket",
            "sweater",
            "hoodie"
        ]

        if any(
            item in items
            for item in heavy_items
        ):

            score -= 2.5

            feedback.append(
                "Heavy clothing may be uncomfortable "
                "in hot weather."
            )

    # ==========================================
    # 4. Rain
    # ==========================================

    if weather == "rainy":

        if "sunglasses" in items:

            score -= 0.5

            feedback.append(
                "Sunglasses may be less practical "
                "during rainy conditions."
            )

    # ==========================================
    # 5. Time of Day
    # ==========================================

    if (
        time_of_day in {
            "evening",
            "night"
        }
        and "sunglasses" in items
    ):

        score -= 1.5

        feedback.append(
            "Sunglasses are generally unnecessary "
            "in the evening or at night."
        )

    # ==========================================
    # 6. Remove Duplicate Feedback
    # ==========================================

    feedback = list(
        dict.fromkeys(feedback)
    )

    # ==========================================
    # 7. Final Score
    # ==========================================

    final_score = round(
        max(
            1.0,
            min(
                10.0,
                score
            )
        ),
        1
    )

    # ==========================================
    # 8. Positive Feedback
    # ==========================================

    if not feedback:

        feedback.append(
            "The outfit is suitable for "
            "the selected occasion and conditions."
        )

    return (
        final_score,
        feedback
    )

In [5]:
# ==========================================
# 3. Test the Suitability System
# ==========================================

detected_clothing_labels = [
    "t-shirt",
    "shorts"
]

user_occasion = "Work"
user_time = "Morning"
user_weather = "Sunny"
user_temperature = 24

suitability_score, feedback_list = (
    evaluate_outfit_suitability(
        detected_clothing_labels,
        user_occasion,
        user_time,
        user_weather,
        user_temperature
    )
)

print("\n========================================")
print("       OUTFIT SUITABILITY RESULT")
print("========================================")

print(
    f"Occasion    : {user_occasion}"
)

print(
    f"Time        : {user_time}"
)

print(
    f"Weather     : {user_weather}"
)

print(
    f"Temperature : {user_temperature}°C"
)

print(
    f"Items       : {detected_clothing_labels}"
)

print(
    f"\nSuitability Score: "
    f"{suitability_score} / 10"
)

print("\nFeedback:")

for feedback in feedback_list:

    print(
        f"• {feedback}"
    )

print("========================================")


       OUTFIT SUITABILITY RESULT
Occasion    : Work
Time        : Morning
Weather     : Sunny
Temperature : 24°C
Items       : ['t-shirt', 'shorts']

Suitability Score: 6.5 / 10

Feedback:
• Shorts may be too casual for work or university.


### Suitability Evaluation Results

The final suitability score and recommendations are generated based on the detected clothing items and the selected environmental conditions.